In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from factor_analyzer import FactorAnalyzer as FA
from scipy.stats import zscore
from sklearn.decomposition import PCA

## 10.1

In [ ]:
# 程序文件 ex12_1.py

a = np.loadtxt("data/12_1.txt")
b = zscore(a, ddof=1)  # 数据标准化
md = PCA().fit(b)  # 构造并拟合模型
print("特征值为：", md.explained_variance_)
print("各主成分贡献率：", md.explained_variance_ratio_)
xs1 = md.components_  # 提出各主成分系数，每行是一个主成分
print("各主成分系数:\n", xs1)
check = xs1.sum(axis=1, keepdims=True)  # 计算各个主成分系数的和
xs2 = xs1 * np.sign(check)  # 调整主成分系数，和为负时乘以 -1
print("调整后的主成分系数：", xs2)

## 10.2

In [ ]:
# 程序文件 ex12_2.py

a = np.loadtxt("data/12_2.txt")
mu = a.mean(axis=0)  # 逐列求均值
s = a.std(axis=0, ddof=1)  # 逐列求标准差
b = (a - mu) / s  # 数据标准化
r = np.corrcoef(b[:, :-1].T)  # 计算相关系数矩阵
md1 = PCA().fit(b[:, :-1])  # 构造并拟合模型
print("特征值为：", md1.explained_variance_)
print("各主成分贡献率：", md1.explained_variance_ratio_)
xs = md1.components_  # 提出各主成分系数，每行是一个主成分
print("主成分系数：\n", np.round(xs, 4))
print("累积贡献率：", np.cumsum(md1.explained_variance_ratio_))

n = 3  # 选定主成分的个数
f = b[:, :-1] @ (xs[:n, :].T)  # 主成分的得分
d2 = {"y": a[:, -1], "x": a[:, :-1]}
md2 = sm.formula.ols("y~x", d2).fit()  # 原始数据线性回归
d3 = {"y": a[:, -1], "z": f}
md3 = sm.formula.ols("y~z", d3).fit()  # 对主成分的回归方程
xs3 = md3.params  # 提取主成分回归方程的系数
xs40 = xs3[0] - sum(xs3[1:] @ xs[:n, :] * mu[:-1] / s[:-1])  # 常数项
xs4 = xs3[1:] @ xs[:n, :] / s[:-1]  # 原始变量回归方程的其他系数
print("回归方程的常数项：", round(xs40, 4))
print("回归方程的其他系数：", np.round(xs4, 4))
print("直接回归的残差方差：", md2.mse_resid)
print("主成分回归的残差方差：", md3.mse_resid)

## 10.3

In [ ]:
# 程序文件 ex12_3.py

a = pd.read_excel("data/12_3.xlsx", header=None)
b = a.values
c = zscore(b, ddof=1)  # 数据标准化
md1 = PCA().fit(c)  # 构造并拟合模型
print("特征值为：", md1.explained_variance_)
r1 = md1.explained_variance_ratio_  # 提取各主成分的贡献率
print("各主成分贡献率：", r1)
xs1 = md1.components_  # 提出各主成分系数，每行是一个主成分
print("主成分系数：\n", np.round(xs1, 4))
print("累积贡献率：", np.cumsum(r1))
n1 = 4  # 选取主成分的个数
df1 = c @ (xs1[:n1, :].T)  # 计算主成分得分
g1 = df1 @ r1[:n1]  # 计算综合评价得分
print("主成分评价得分", np.round(g1, 4))
ind1 = np.argsort(-g1)  # 计算从大到小的地址
ind11 = np.zeros(15)
ind11[ind1] = np.arange(1, 16)
print("排序结果：", ind11)
print("---------------")

n = b.shape[0]
K = np.zeros((n, n))  # 初始化
for i in range(n):
    for j in range(n):
        K[i, j] = (1 + c[i] @ c[j]) ** 3
In = np.ones((n, n)) / n
Kw = K - K @ In - In @ K + In @ K @ In
md2 = PCA().fit(Kw)  # 构造并拟合模型
print("特征值为：", md2.explained_variance_)
r2 = md2.explained_variance_ratio_  # 提取各主成分的贡献率
print("各主成分贡献率：", r2)
xs2 = md2.components_  # 提出各主成分系数，每行是一个主成分
print("主成分系数：\n", np.round(xs2, 4))
print("累积贡献率：", np.cumsum(r2))
n2 = 2  # 选取主成分的个数
df2 = Kw @ (xs2[:n2, :].T)  # 计算主成分得分
g2 = df2 @ r2[:n2]  # 计算综合评价得分
print("核主成分评价得分：", np.round(g2))
ind2 = np.argsort(-g2)  # 计算从大到小的地址
ind22 = np.zeros(15)
ind22[ind2] = np.arange(1, 16)
print("排序结果：", ind22)

## 10.4

In [ ]:
# 程序文件ex12_4.py

r = np.array([[1, 1 / 5, -1 / 5], [1 / 5, 1, -2 / 5], [-1 / 5, -2 / 5, 1]])
val, vec = np.linalg.eig(r)  # 求相关系数阵的特征值和特征向量
A0 = vec * np.sqrt(val)  # 利用广播运算求载荷矩阵
print("特征值:", val, "\n载荷矩阵：\n", A0, "\n----------")
num = int(input("请输入选择公共因子的个数："))
A = A0[:, :num]  # 提出num个因子的载荷矩阵
Ac = np.sum(A**2, axis=0)  # 逐列元素求和，求信息贡献
Ar = np.sum(A**2, axis=1)  # 逐行元素求和，求共同度
print("对x的贡献为：", Ac)
print("共同度为：", Ar)

## 10.5

In [ ]:
# 程序文件 ex12_5_1.py

c = np.loadtxt("data/12_5_1.txt")
d = zscore(c, ddof=1)  # 数据标准化
r = np.corrcoef(d.T)  # 求相关系数矩阵
val, vec = np.linalg.eig(r)
cs = np.cumsum(val)  # 求特征值的累加和
rate = val / cs[-1]  # 求贡献率
srate = sorted(rate, reverse=True)
print("特征值为：", val, "\n贡献率为：", srate)
fa = FA(3, rotation="varimax")  # 构建模型
fa.fit(d)  # 求解方差最大的模型
A = fa.loadings_  # 提取载荷矩阵
gx = np.sum(A**2, axis=0)  # 计算信息贡献
s2 = 1 - np.sum(A**2, axis=1)  # 计算特殊方差
ss = np.linalg.inv(np.diag(s2))
f = ss @ A @ np.linalg.inv(A.T @ ss @ A)  # 计算因子得分函数系数
df = d @ f  # 计算因子得分
pj = df @ gx / sum(gx)  # 计算评价值
print("载荷矩阵为：\n", np.round(A, 4))
print("特殊方差为：", np.round(s2, 4))
print("各因子的方差贡献：", np.round(gx, 4))
print("评价值为：\n", pj)
ind0 = np.argsort(-pj)  # 从大到小的排名地址
ind = np.zeros(17)
ind[ind0] = np.arange(1, 18)
print("排名次序为：", ind)
F = pd.ExcelWriter("data/12_5_2.xlsx")
pd.DataFrame(r).to_excel(F)
pd.DataFrame(df).to_excel(F, "Sheet2", index=False)
pd.DataFrame(pj).to_excel(F, "Sheet3", index=False)
F.save()

In [ ]:
# 程序文件 ex12_5.py

c = np.loadtxt("data/12_5_1.txt")
d = zscore(c, ddof=1)  # 数据标准化
r = np.corrcoef(d.T)  # 求相关系数矩阵
val, vec = np.linalg.eig(r)
cs = np.cumsum(val)  # 求特征值的累加和
rate = val / cs[-1]  # 求贡献率
print("特征值为：", val, "\n贡献率为：", rate)
fa = FA(3, rotation="varimax")  # 构建模型
fa.fit(d)  # 求解方差最大的模型
A = fa.components_.T  # 提取载荷矩阵
gx = np.sum(A**2, axis=0)  # 计算信息贡献
s2 = 1 - np.sum(A**2, axis=1)  # 计算特殊方差
ss = np.linalg.inv(np.diag(s2))
df = d @ ss @ A @ np.linalg.inv(A.T @ ss @ A)
print("载荷矩阵为：\n", A)
print("特殊方差为：\n", s2)
pj = df @ gx / sum(gx)  # 计算综合得分